In [1]:
import zipfile 

zipfile.ZipFile("CDR_Data.zip").extractall("CDR_Data")

## What is in `CDR_Data`

BioCreative V Chemical–Disease Relation (CDR) corpus: **1,500 PubMed articles** with chemical mentions, disease mentions, and document-level chemical-induced disease (CID) relations.

The zip has its own `CDR_Data` folder, so after the cell above the files live in `CDR_Data/CDR_Data/`. `__MACOSX/` is Mac archive metadata and is ignored.

| Folder | Role |
| --- | --- |
| `CDR.Corpus.v010516` | Gold train / development / test |
| `tmChem.TestSet` | tmChem chemical mentions on the **test** articles |
| `DNorm.TestSet` | DNorm disease mentions on the **test** articles |

Each split is stored twice: **PubTator** (plain text) and **BioC XML** (same content). The cells below read PubTator.

In [2]:
from pathlib import Path

import pandas as pd
from IPython.display import display

ROOT = Path("CDR_Data/CDR_Data")

rows = []
for path in sorted(ROOT.rglob("*")):
    if not path.is_file() or path.name.startswith("."):
        continue
    rows.append({
        "folder": path.parent.name,
        "file": path.name,
        "size_mb": round(path.stat().st_size / 1e6, 2),
    })

files = pd.DataFrame(rows)
files

,folder,file,size_mb
0,CDR_Data,BC5CDR.corpus.pdf,0.77
1,CDR_Data,BC5CDR.overview.pdf,0.25
2,CDR_Data,BC5CDR.presentation.pdf,16.67
3,CDR.Corpus.v010516,CDR_DevelopmentSet.BioC.xml,2.54
4,CDR.Corpus.v010516,CDR_DevelopmentSet.PubTator.txt,1.13
5,CDR.Corpus.v010516,CDR_TestSet.BioC.xml,2.67
6,CDR.Corpus.v010516,CDR_TestSet.PubTator.txt,1.17
7,CDR.Corpus.v010516,CDR_TrainingSet.BioC.xml,2.52
8,CDR.Corpus.v010516,CDR_TrainingSet.PubTator.txt,1.13
9,DNorm.TestSet,TestSet.DNorm.BioC.xml,1.56


## PubTator layout

One article is a block of lines, then a blank line.

```
PMID|t|title
PMID|a|abstract
PMID    start    end    mention    Chemical|Disease    MeSH
PMID    CID    chemical MeSH    disease MeSH
```

Character offsets index `title + " " + abstract` (one space between the two). A MeSH id of `-1` means that mention was not normalized to a concept. Some mentions carry several concepts in one span, with ids separated by `|` (`D006470|D003556` for "hemorrhagic cystitis"). A `CID` line is one document-level relation between a chemical concept and a disease concept. It is not attached to a single sentence, and most concept pairs in an article are not relations.

In [3]:
def parse_pubtator(path):
    """Read a PubTator file into a list of article dicts."""
    docs = []
    doc = None

    def flush():
        nonlocal doc
        if doc is not None:
            docs.append(doc)
            doc = None

    for raw in Path(path).read_text(encoding="utf-8").splitlines():
        line = raw.strip()
        if not line:
            flush()
            continue

        if "\t" not in line:
            pmid, kind, text = line.split("|", 2)
            if doc is None or doc["pmid"] != pmid:
                flush()
                doc = {
                    "pmid": pmid,
                    "title": "",
                    "abstract": "",
                    "mentions": [],
                    "relations": [],
                }
            if kind == "t":
                doc["title"] = text
            elif kind == "a":
                doc["abstract"] = text
            continue

        cols = line.split("\t")
        if doc is None or doc["pmid"] != cols[0]:
            flush()
            doc = {
                "pmid": cols[0],
                "title": "",
                "abstract": "",
                "mentions": [],
                "relations": [],
            }
        if len(cols) == 4 and cols[1] == "CID":
            doc["relations"].append({
                "chemical_mesh": cols[2],
                "disease_mesh": cols[3],
            })
        else:
            mesh = cols[5] if len(cols) > 5 else None
            doc["mentions"].append({
                "start": int(cols[1]),
                "end": int(cols[2]),
                "text": cols[3],
                "type": cols[4],
                "mesh": mesh,
                "mesh_ids": [] if not mesh else mesh.split("|"),
            })

    flush()
    return docs


GOLD = {
    "train": ROOT / "CDR.Corpus.v010516" / "CDR_TrainingSet.PubTator.txt",
    "dev": ROOT / "CDR.Corpus.v010516" / "CDR_DevelopmentSet.PubTator.txt",
    "test": ROOT / "CDR.Corpus.v010516" / "CDR_TestSet.PubTator.txt",
}

splits = {name: parse_pubtator(path) for name, path in GOLD.items()}
{name: len(docs) for name, docs in splits.items()}

{'train': 500, 'dev': 500, 'test': 500}

In [4]:
def article_text(doc):
    # PubTator offsets count one space between the title and the abstract.
    return doc["title"] + " " + doc["abstract"]


def summarize(name, docs):
    mentions = [m for doc in docs for m in doc["mentions"]]
    relations = [r for doc in docs for r in doc["relations"]]
    offset_ok = sum(
        article_text(doc)[m["start"]:m["end"]] == m["text"]
        for doc in docs for m in doc["mentions"]
    )
    return {
        "split": name,
        "articles": len(docs),
        "mentions": len(mentions),
        "chemical_mentions": sum(m["type"] == "Chemical" for m in mentions),
        "disease_mentions": sum(m["type"] == "Disease" for m in mentions),
        "unnormalized_mentions": sum(m["mesh_ids"] in ([], ["-1"]) for m in mentions),
        "multi_mesh_mentions": sum(len(m["mesh_ids"]) > 1 for m in mentions),
        "cid_relations": len(relations),
        "articles_with_no_cid": sum(len(doc["relations"]) == 0 for doc in docs),
        "offsets_match_text": offset_ok == len(mentions),
    }


summary = pd.DataFrame(summarize(name, docs) for name, docs in splits.items())
summary

,split,articles,mentions,chemical_mentions,disease_mentions,unnormalized_mentions,multi_mesh_mentions,cid_relations,articles_with_no_cid,offsets_match_text
0,train,500,9385,5203,4182,76,98,1038,0,True
1,dev,500,9591,5347,4244,59,100,1012,0,True
2,test,500,9809,5385,4424,91,75,1066,0,True


## One training article

PMID 227508 is the first training article. It has many chemical and disease mentions and only one CID relation: MeSH `D008750` (alpha-methyldopa) causes MeSH `D007022` (hypotensive). Mentions such as naloxone and clonidine are annotated, but they are not the labeled relation. `nalozone` is a typo and stays at MeSH `-1`.

In [5]:
doc = splits["train"][0]
text = article_text(doc)

print(doc["pmid"])
print("TITLE:", doc["title"])
print()
print(doc["abstract"])
print()
print("CID relations:")
for rel in doc["relations"]:
    print(f"  {rel['chemical_mesh']}  ->  {rel['disease_mesh']}")

mention_rows = []
for m in doc["mentions"]:
    mention_rows.append({
        "where": "title" if m["end"] <= len(doc["title"]) else "abstract",
        "type": m["type"],
        "text": m["text"],
        "mesh": m["mesh"],
        "start": m["start"],
        "end": m["end"],
        "offset_ok": text[m["start"]:m["end"]] == m["text"],
    })

pd.DataFrame(mention_rows)

227508
TITLE: Naloxone reverses the antihypertensive effect of clonidine.

In unanesthetized, spontaneously hypertensive rats the decrease in blood pressure and heart rate produced by intravenous clonidine, 5 to 20 micrograms/kg, was inhibited or reversed by nalozone, 0.2 to 2 mg/kg. The hypotensive effect of 100 mg/kg alpha-methyldopa was also partially reversed by naloxone. Naloxone alone did not affect either blood pressure or heart rate. In brain membranes from spontaneously hypertensive rats clonidine, 10(-8) to 10(-5) M, did not influence stereoselective binding of [3H]-naloxone (8 nM), and naloxone, 10(-8) to 10(-4) M, did not influence clonidine-suppressible binding of [3H]-dihydroergocryptine (1 nM). These findings indicate that in spontaneously hypertensive rats the effects of central alpha-adrenoceptor stimulation involve activation of opiate receptors. As naloxone and clonidine do not appear to interact with the same receptor site, the observed functional antagonism suggest

,where,type,text,mesh,start,end,offset_ok
0,title,Chemical,Naloxone,D009270,0,8,True
1,title,Chemical,clonidine,D003000,49,58,True
2,abstract,Disease,hypertensive,D006973,93,105,True
3,abstract,Chemical,clonidine,D003000,181,190,True
4,abstract,Chemical,nalozone,-1,244,252,True
5,abstract,Disease,hypotensive,D007022,274,285,True
6,abstract,Chemical,alpha-methyldopa,D008750,306,322,True
7,abstract,Chemical,naloxone,D009270,354,362,True
8,abstract,Chemical,Naloxone,D009270,364,372,True
9,abstract,Disease,hypertensive,D006973,469,481,True


## How relations sit inside an article

At document level the candidates are concept pairs, not raw strings. Take every normalized chemical MeSH id in the article and every normalized disease MeSH id (split on `|` when one mention has several). Their Cartesian product is the set of possible CID pairs. Gold `CID` lines are the positive pairs. `-1` mentions never enter that set, because they have no concept id.

In [6]:
def concept_ids(doc, entity_type):
    return {
        mesh
        for m in doc["mentions"]
        if m["type"] == entity_type
        for mesh in m["mesh_ids"]
        if mesh != "-1"
    }


def pair_frame(docs):
    rows = []
    for doc in docs:
        chemicals = concept_ids(doc, "Chemical")
        diseases = concept_ids(doc, "Disease")
        candidates = {(c, d) for c in chemicals for d in diseases}
        gold = {(r["chemical_mesh"], r["disease_mesh"]) for r in doc["relations"]}
        rows.append({
            "pmid": doc["pmid"],
            "chemicals": len(chemicals),
            "diseases": len(diseases),
            "candidate_pairs": len(candidates),
            "gold_cid": len(gold),
            "gold_inside_candidates": len(gold & candidates),
        })
    return pd.DataFrame(rows)


pair_stats = []
for name, docs in splits.items():
    frame = pair_frame(docs)
    pair_stats.append({
        "split": name,
        "candidate_pairs": int(frame["candidate_pairs"].sum()),
        "gold_cid": int(frame["gold_cid"].sum()),
        "gold_missing_from_candidates": int((frame["gold_cid"] - frame["gold_inside_candidates"]).sum()),
        "median_candidates_per_article": float(frame["candidate_pairs"].median()),
        "max_candidates_in_one_article": int(frame["candidate_pairs"].max()),
        "articles_with_zero_candidates": int((frame["candidate_pairs"] == 0).sum()),
    })

pd.DataFrame(pair_stats)

,split,candidate_pairs,gold_cid,gold_missing_from_candidates,median_candidates_per_article,max_candidates_in_one_article,articles_with_zero_candidates
0,train,5432,1038,0,7.0,104,0
1,dev,5261,1012,0,7.0,91,0
2,test,5405,1066,0,8.0,117,0


In [7]:
train_pairs = pair_frame(splits["train"])
train_pairs["gold_cid"].value_counts().sort_index().rename("articles").to_frame()

,articles
gold_cid,
1,250
2,130
3,46
4,31
5,16
6,18
7,3
8,3
9,1


## Which concepts show up most often in training

Counts below are **articles** that mention the concept at least once, not raw string occurrences. The same drug is often written several times in one abstract.

In [8]:
def concept_article_counts(docs, entity_type, n=15):
    counts = {}
    examples = {}
    for doc in docs:
        seen = {}
        for m in doc["mentions"]:
            if m["type"] != entity_type:
                continue
            for mesh in m["mesh_ids"]:
                if mesh != "-1":
                    seen.setdefault(mesh, m["text"])
        for mesh, surface in seen.items():
            counts[mesh] = counts.get(mesh, 0) + 1
            examples.setdefault(mesh, surface)
    ranked = sorted(counts.items(), key=lambda item: item[1], reverse=True)[:n]
    return pd.DataFrame(
        [{"mesh": mesh, "example": examples[mesh], "articles": count} for mesh, count in ranked]
    )


print("Chemicals")
display(concept_article_counts(splits["train"], "Chemical"))
print("Diseases")
display(concept_article_counts(splits["train"], "Disease"))

Chemicals


,mesh,example,articles
0,D004298,dopamine,27
1,D003042,cocaine,19
2,D002118,Calcium,17
3,D003404,creatinine,16
4,D002945,Cisplatin,13
5,D006220,haloperidol,13
6,D004317,adriamycin,13
7,D011188,potassium,13
8,D016559,tacrolimus,13
9,D000109,acetylcholine,12


Diseases


,mesh,example,articles
0,D064420,toxicity,47
1,D012640,seizures,41
2,D007674,nephropathy,39
3,D007022,hypotensive,34
4,D006973,hypertensive,32
5,D010146,pain,27
6,D009369,tumor,27
7,D003866,depression,22
8,D056486,hepatitis,22
9,D009422,neuropathy,19


## tmChem and DNorm on the test set

These two folders are **system output**, not extra gold labels. tmChem marks chemicals. DNorm marks diseases. Neither file contains `CID` lines. Identifier style also differs from the gold corpus: gold uses a bare MeSH id (`D009270`), while the tool files often use a prefix (`MESH:D015738`, and sometimes `CHEBI:...`). tmChem also leaves some mentions with no id at all; those lines end after the type, and the text is often a fragment such as `bort`.

In [9]:
tmchem = parse_pubtator(ROOT / "tmChem.TestSet" / "TestSet.tmChem.PubTator.txt")
dnorm = parse_pubtator(ROOT / "DNorm.TestSet" / "TestSet.DNorm.PubTator.txt")
gold_test = splits["test"]


def id_kind(mesh):
    if mesh in (None, ""):
        return "missing"
    if mesh == "-1":
        return "-1"
    if ":" in mesh:
        return mesh.split(":", 1)[0]
    return "bare"


def source_row(name, docs):
    mentions = [m for doc in docs for m in doc["mentions"]]
    kinds = pd.Series(id_kind(m["mesh"]) for m in mentions).value_counts()
    return {
        "source": name,
        "articles": len(docs),
        "mentions": len(mentions),
        "relations": sum(len(doc["relations"]) for doc in docs),
        "id_styles": ", ".join(f"{k}:{int(v)}" for k, v in kinds.items()),
    }


compare = pd.DataFrame([
    source_row("gold test", gold_test),
    source_row("tmChem", tmchem),
    source_row("DNorm", dnorm),
])
display(compare)

gold_ids = {doc["pmid"] for doc in gold_test}
print("same articles as gold test:",
      {doc["pmid"] for doc in tmchem} == gold_ids,
      {doc["pmid"] for doc in dnorm} == gold_ids)

,source,articles,mentions,relations,id_styles
0,gold test,500,9809,1066,"bare:9718, -1:91"
1,tmChem,500,5107,0,"MESH:3932, bare:609, missing:405, CHEBI:161"
2,DNorm,500,4277,0,MESH:4277


same articles as gold test: True True


In [10]:
by_id = {
    "gold": {doc["pmid"]: doc for doc in gold_test},
    "tmChem": {doc["pmid"]: doc for doc in tmchem},
    "DNorm": {doc["pmid"]: doc for doc in dnorm},
}
pmid = gold_test[0]["pmid"]
print(pmid)
print(by_id["gold"][pmid]["title"])

for name in ("gold", "tmChem", "DNorm"):
    doc = by_id[name][pmid]
    rows = [
        {"source": name, "type": m["type"], "text": m["text"], "mesh": m["mesh"]}
        for m in doc["mentions"]
    ]
    print()
    display(pd.DataFrame(rows))

8701013
Famotidine-associated delirium. A series of six cases.



,source,type,text,mesh
0,gold,Chemical,Famotidine,D015738
1,gold,Disease,delirium,D003693
2,gold,Chemical,Famotidine,D015738
3,gold,Disease,ulcers,D014456
4,gold,Disease,delirium,D003693
5,gold,Chemical,famotidine,D015738
6,gold,Chemical,famotidine,D015738
7,gold,Disease,delirium,D003693
8,gold,Chemical,famotidine,D015738
9,gold,Chemical,famotidine,D015738


,source,type,text,mesh
0,tmChem,Chemical,Famotidine,MESH:D015738
1,tmChem,Chemical,Famotidine,MESH:D015738
2,tmChem,Chemical,histamine,MESH:D006632
3,tmChem,Chemical,famotidine,MESH:D015738
4,tmChem,Chemical,famotidine,MESH:D015738
5,tmChem,Chemical,famotidine,MESH:D015738
6,tmChem,Chemical,famotidine,MESH:D015738
7,tmChem,Chemical,famotidine,MESH:D015738


,source,type,text,mesh
0,DNorm,Disease,delirium,MESH:D003693
1,DNorm,Disease,ulcers,MESH:D014456
2,DNorm,Disease,delirium,MESH:D003693
3,DNorm,Disease,delirium,MESH:D003693
